# 🍽️ 경기도 휴게음식점 데이터 수집

경기도 OpenAPI를 사용하여 휴게음식점 데이터를 수집하고 PostgreSQL DB에 저장하는 노트북입니다.

## 📋 작업 흐름

1. **패키지 설치** - 필요한 라이브러리 설치
2. **환경 설정** - API 키, DB 연결 정보 로드
3. **함수 정의** - 데이터 변환, API 호출, DB 저장 함수
4. **테스트** - 환경 설정 및 API 연결 확인
5. **실행** - 전체 데이터 수집 및 저장

## ⚙️ 사전 준비

- `.env` 파일에 다음 환경 변수 설정:
  - `RESTAURANT_API_KEY`: 경기도 OpenAPI 인증키
  - `PGHOST`, `PGPORT`, `PGDATABASE`, `PGUSER`, `PGPASSWORD`: PostgreSQL 연결 정보
- PostgreSQL 서버 실행
- `locallink.gg_restaurants` 테이블 생성

# 📦 필수 패키지 설치

데이터 수집에 필요한 Python 라이브러리를 설치합니다.

In [17]:
%pip install pyproj requests pandas IPython datetime display python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [18]:
%pip install pandas sqlalchemy psycopg2 requests

Note: you may need to restart the kernel to use updated packages.


# 📦 라이브러리 Import 및 기본 설정

필요한 라이브러리를 불러오고 API 및 DB 연결 정보를 설정합니다.

**주요 설정값:**
- `API_URL`: 경기도 휴게음식점 OpenAPI 엔드포인트
- `PAGE_SIZE`: 한 번에 가져올 데이터 개수 (최대 1000개 권장)
- `SLEEP_SEC`: API 호출 간 대기 시간 (서버 부하 방지)
- `PG_DSN`: PostgreSQL 데이터베이스 연결 정보 (.env 파일에서 로드)

# ✅ 테스트 및 디버깅

실제 데이터 수집을 실행하기 전에 환경 설정과 API 연결을 테스트합니다.

**확인 항목:**
1. ✓ API_KEY가 .env 파일에서 제대로 로드되었는지
2. ✓ PostgreSQL 서버에 연결 가능한지
3. ✓ API가 정상적으로 응답하는지

모든 테스트가 통과하면 아래의 `main()` 함수를 실행하세요!

In [16]:
# ── 디버깅: 환경 변수 및 API 연결 확인 ─────────────────────────────────
print("=" * 60)
print("환경 변수 확인")
print("=" * 60)
print(f"API_KEY 설정됨: {bool(API_KEY)}")
print(f"API_URL: {API_URL}")

# DB 연결 테스트
print("\nDB 연결 테스트...")
try:
    test_conn = psycopg2.connect(PG_DSN)
    print("✓ DB 연결 성공")
    test_conn.close()
except Exception as e:
    print(f"✗ DB 연결 실패: {e}")

# API 호출 테스트 (1페이지만)
print("\nAPI 호출 테스트...")
try:
    total, rows = fetch_page(1, 100)  # 작은 크기로 테스트
    print(f"✓ API 호출 성공")
    print(f"  - 총 건수: {total}")
    print(f"  - 반환된 행: {len(rows)}")
    if rows:
        print(f"  - 첫 번째 행: {list(rows[0].keys())[:5]}...")
except Exception as e:
    print(f"✗ API 호출 실패: {e}")

print("\n" + "=" * 60)
print("main() 함수 실행")
print("=" * 60)
# main()  # 주석 해제하여 실행
print("※ 위 테스트가 모두 성공하면 아래를 실행하세요:")
print("main()")


환경 변수 확인
API_KEY 설정됨: True
API_URL: https://openapi.gg.go.kr/Resrestrtcvnstr

DB 연결 테스트...
✓ DB 연결 성공

API 호출 테스트...
✓ API 호출 성공
  - 총 건수: 15854
  - 반환된 행: 100
  - 첫 번째 행: ['SIGUN_NM', 'SIGUN_CD', 'BIZPLC_NM', 'LICENSG_DE', 'BSN_STATE_NM']...

main() 함수 실행
※ 위 테스트가 모두 성공하면 아래를 실행하세요:
main()


In [ ]:
import os
import math
import time
import requests
import psycopg2
from psycopg2.extras import execute_values
from datetime import datetime
from dotenv import load_dotenv, find_dotenv


# ── 설정 ───────────────────────────────────────────────────────────
API_URL = "https://openapi.gg.go.kr/Resrestrtcvnstr"  # 휴게음식점 OpenAPI 엔드포인트
PAGE_SIZE = 1000       # 성능을 위해 최대치 권장(문서 기본은 100)
SLEEP_SEC = 0.2        # 호출 간 딜레이(기관 트래픽 정책 배려)

load_dotenv(find_dotenv())

# API 설정
API_KEY = os.getenv("RESTAURANT_API_KEY")

PG_DSN = (
    f"host={os.getenv('PGHOST','localhost')} "
    f"port={os.getenv('PGPORT','5432')} "
    f"dbname={os.getenv('PGDATABASE','yourdb')} "
    f"user={os.getenv('PGUSER','youruser')} "
    f"password={os.getenv('PGPASSWORD','yourpassword')}"
)

# 🔧 유틸리티 함수

API 응답 데이터를 안전하게 변환하는 헬퍼 함수들입니다.

- `to_date()`: 날짜 문자열을 date 객체로 변환 (YYYYMMDD, YYYY-MM-DD 등 형식 자동 인식)
- `to_float()`: 문자열을 실수로 변환 (실패 시 None 반환)
- `to_int()`: 문자열을 정수로 변환 (실패 시 None 반환)

In [ ]:
# ── 유틸: 안전 변환 ────────────────────────────────────────────────
def to_date(s):
    """YYYY-MM-DD, YYYYMMDD 등 혼재 대응"""
    if not s:
        return None
    s = str(s).strip()
    for fmt in ("%Y-%m-%d", "%Y%m%d", "%Y.%m.%d"):
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            continue
    return None

def to_float(s):
    try:
        return float(str(s).strip())
    except Exception:
        return None

def to_int(s):
    try:
        return int(float(str(s).strip()))
    except Exception:
        return None

# 🌐 API 호출 함수

경기도 OpenAPI에서 휴게음식점 데이터를 가져오는 함수입니다.

**fetch_page(p_index, p_size, sigun_nm, sigun_cd)**
- API 한 페이지를 호출하여 **(총건수, 데이터 배열)** 반환
- JSON 응답 구조: `{"Resrestrtcvnstr": [{"head": {...}, "row": [{...}]}]}`
- 시군명(sigun_nm) 또는 시군코드(sigun_cd)로 필터링 가능

In [ ]:
def fetch_page(p_index=1, p_size=PAGE_SIZE, sigun_nm=None, sigun_cd=None):
    """
    한 페이지 호출하여 (총건수, row배열) 반환
    응답 구조 예시: {"Resrestrtcvnstr":[{"head":[... {"list_total_count":...} ...]}, {"row":[{...}, ...]}]}
    """
    params = {
        "KEY": API_KEY,
        "Type": "json",     # json 권장
        "pIndex": p_index,
        "pSize": p_size
    }
    if sigun_nm:
        params["SIGUN_NM"] = sigun_nm
    if sigun_cd:
        params["SIGUN_CD"] = sigun_cd

    r = requests.get(API_URL, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    root = data.get("Resrestrtcvnstr")
    if not root or not isinstance(root, list):
        return 0, []

    # 총건수 파싱
    list_total_count = None
    if root and "head" in root[0]:
        for h in root[0]["head"]:
            if isinstance(h, dict):
                for k, v in h.items():
                    if k.lower() == "list_total_count":
                        list_total_count = int(v)

    # row 수집
    rows = []
    for bucket in root:
        if isinstance(bucket, dict) and "row" in bucket:
            rows.extend(bucket["row"])

    if list_total_count is None:
        list_total_count = len(rows)

    return list_total_count, rows

# 💾 데이터베이스 저장 함수

API 응답 데이터를 PostgreSQL DB에 저장하는 함수들입니다.

**transform_row(r)**
- API 응답의 각 행을 DB 테이블 컬럼 순서에 맞게 변환
- 43개 필드를 튜플로 반환

**insert_rows(conn, rows)**
- 여러 행을 한 번에 DB에 INSERT
- `execute_values`를 사용하여 성능 최적화 (page_size=5000)

In [ ]:
def transform_row(r):
    """
    API의 46개 필드를 우리 테이블의 영문 컬럼 순서에 맞춰 매핑.
    필드명은 명세서의 Key(대문자) 기준.
    """
    return (
        r.get("SIGUN_CD"),
        r.get("BIZPLC_NM"),
        to_date(r.get("LICENSG_DE")),

        r.get("REFINE_ROADNM_ADDR"),
        r.get("REFINE_LOTNO_ADDR"),
        r.get("REFINE_ZIP_CD"),

        to_float(r.get("REFINE_WGS84_LAT")),
        to_float(r.get("REFINE_WGS84_LOGT")),

        to_date(r.get("LICENSG_CANCL_DE")),

        r.get("BSN_STATE_DIV_CD"),
        r.get("UNITY_BSN_STATE_DIV_CD"),
        r.get("UNITY_BSN_STATE_NM"),
        r.get("BSN_STATE_NM"),

        to_date(r.get("CLSBIZ_DE")),
        to_date(r.get("SUSPNBIZ_BEGIN_DE")),
        to_date(r.get("SUSPNBIZ_END_DE")),
        to_date(r.get("REOPENBIZ_DE")),

        r.get("LOCPLC_FACLT_TELNO"),
        r.get("LOCPLC_AR_INFO"),
        r.get("BIZCOND_DIV_NM_INFO"),

        to_float(r.get("X_CRDNT_VL")),
        to_float(r.get("Y_CRDNT_VL")),

        r.get("SANITTN_BIZCOND_NM"),
        to_int(r.get("MALE_ENFLPSN_CNT")),
        to_int(r.get("FEMALE_ENFLPSN_CNT")),

        r.get("BSNSITE_CIRCUMFR_DIV_NM"),
        r.get("GRAD_DIV_NM"),
        r.get("GRAD_FACLT_DIV_NM"),

        to_int(r.get("TOT_EMPLY_CNT")),
        to_int(r.get("HEADOFC_EMPLY_CNT")),
        to_int(r.get("FACTRY_OFCRK_DUT_EMPLY_CNT")),
        to_int(r.get("FACTRY_SALE_DUT_EMPLY_CNT")),
        to_int(r.get("FACTRY_PRODCTN_DUT_EMPLY_CNT")),

        r.get("BULDNG_POSESN_DIV_NM"),
        to_float(r.get("ASSURNC_AMT")),
        to_float(r.get("MTRENT_AMT")),

        r.get("MULTI_USE_BIZESTBL_YN"),
        r.get("FACLT_TOT_SCALE_INFO"),
        r.get("TRADITN_BIZESTBL_APPONT_NO"),
        r.get("TRADITN_BIZESTBL_CHIEF_FOOD_NM"),
        r.get("HMPG_URL"),

        r.get("SIGUN_NM"),
        # loaded_at → DEFAULT now()로 DB에서 자동 세팅
    )

def insert_rows(conn, rows):
    sql = """
        INSERT INTO locallink.gg_restaurants (
            sigun_cd, bizplc_nm, licensg_de,
            refine_roadnm_addr, refine_lotno_addr, refine_zip_cd,
            refine_wgs84_lat, refine_wgs84_logt,
            licensg_cancl_de,
            bsn_state_div_cd, unity_bsn_state_div_cd, unity_bsn_state_nm, bsn_state_nm,
            clsbiz_de, suspnbiz_begin_de, suspnbiz_end_de, reopenbiz_de,
            locplc_faclt_telno, locplc_ar_info, bizcond_div_nm_info,
            x_crdnt_vl, y_crdnt_vl,
            sanittn_bizcond_nm, male_enflpsn_cnt, female_enflpsn_cnt,
            bsnsited_circumfr_div_nm, grad_div_nm, grad_faclt_div_nm,
            tot_emply_cnt, headofc_emply_cnt, factry_ofcrk_dut_emply_cnt,
            factry_sale_dut_emply_cnt, factry_prodctn_dut_emply_cnt,
            buldng_posesn_div_nm, assurnc_amt, mtrent_amt,
            multi_use_bizestbl_yn, faclt_tot_scale_info, traditn_bizestbl_appont_no,
            traditn_bizestbl_chief_food_nm, hmpg_url,
            sigun_nm
        )
        VALUES %s
    """
    data = [transform_row(r) for r in rows]
    with conn.cursor() as cur:
        execute_values(cur, sql, data, page_size=5000)

# 🚀 메인 실행 함수

API 데이터를 전체적으로 가져와서 DB에 저장하는 메인 함수입니다.

**main(sigun_nm, sigun_cd)**
1. 첫 페이지를 호출하여 총 데이터 건수 파악
2. 전체 페이지 수 계산 (총 건수 ÷ PAGE_SIZE)
3. 모든 페이지를 순회하며 데이터 저장
4. 각 페이지마다 0.2초 대기 (서버 부하 방지)

**사용 예시:**
- `main()` - 전체 데이터
- `main(sigun_nm="수원시")` - 수원시만
- `main(sigun_cd="41110")` - 시군코드로 필터링

In [ ]:
def main(sigun_nm=None, sigun_cd=None):
    conn = psycopg2.connect(PG_DSN)
    conn.autocommit = False

    # 1페이지 호출: 총 건수 파악
    total, rows = fetch_page(1, PAGE_SIZE, sigun_nm, sigun_cd)
    print(f"[API] total={total}, page=1, rows={len(rows)}")
    if rows:
        insert_rows(conn, rows)
        conn.commit()

    # 전체 페이지 계산
    pages = max(1, math.ceil(total / PAGE_SIZE))

    # 2페이지 이후 반복
    for p in range(2, pages + 1):
        time.sleep(SLEEP_SEC)
        _, rows = fetch_page(p, PAGE_SIZE, sigun_nm, sigun_cd)
        if not rows:
            break
        insert_rows(conn, rows)
        conn.commit()
        print(f"[API] page={p}/{pages}, rows={len(rows)}")

    conn.close()
    print("Done.")


# ▶️ 데이터 수집 실행

**주의:** PostgreSQL 서비스가 실행 중인지 확인하세요!

위의 테스트 셀에서 모든 항목이 ✓로 표시되면 아래 셀을 실행하여 데이터 수집을 시작합니다.

**실행 시 동작:**
- 경기도 전체 휴게음식점 데이터 수집 (약 15만 건)
- 페이지당 1,000개씩 처리
- 각 페이지마다 진행 상황 출력
- 완료까지 약 5~10분 소요 (네트워크 속도에 따라 다름)

In [20]:
main()

[API] total=15854, page=1, rows=1000
[API] page=2/16, rows=1000
[API] page=3/16, rows=1000
[API] page=4/16, rows=1000
[API] page=5/16, rows=1000
[API] page=6/16, rows=1000
[API] page=7/16, rows=1000
[API] page=8/16, rows=1000
[API] page=9/16, rows=1000
[API] page=10/16, rows=1000
[API] page=11/16, rows=1000
[API] page=12/16, rows=1000
[API] page=13/16, rows=1000
[API] page=14/16, rows=1000
[API] page=15/16, rows=1000
[API] page=16/16, rows=854
Done.
